In [1]:
!pip install -r requirements.txt


In [2]:
from orchestrator_agent_no_rag import MultiAgentNoRAG
import os
from pathlib import Path
from test_cases import TEST_CASES
import json
os.environ["OPENAI_API_KEY"] = Path("open_ai_api_key.txt").read_text().strip()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
agent = MultiAgentNoRAG()


In [4]:
result_traces = []
for case in TEST_CASES: 
    answer,_,trace_info = await agent.answer(case["prompt"])
    # Add the response to the traces
    trace_info["answer"] = answer
    #Append to result traces
    result_traces.append(trace_info)
    print(f"Case {case["id"]} Done")



Case 1 Done
Case 2 Done
Case 3 Done
Case 4 Done
Case 5 Done
Case 6 Done
Case 7 Done
Case 8 Done
Case 9 Done
Case 10 Done
Case 11 Done
Case 12 Done
Case 13 Done
Case 14 Done
Case 15 Done
Case 16 Done
Case 17 Done
Case 18 Done
Case 19 Done
Case 20 Done
Case 21 Done


In [5]:
FILE_NAME = "eval_suite_no_rag.json"
#dump to file 
json.dump(result_traces, open(FILE_NAME, "w"), indent=2, default=str)
#print(result_traces[0]["sources"])
#print(result_traces[2]["sources"])

In [6]:
import pandas as pd


with open(FILE_NAME,"r", encoding="utf-8") as file:
    data = json.load(file)


for i in range(len(data)):
    # Stamp the category of the test case
    data[i]["category"] = TEST_CASES[i]["name"]
    # Stamp the response notes from the test case
    data[i]["expected_response_notes"] = TEST_CASES[i]["expectations"]["response_notes"]
    

    # Check guardrails
    guard = data[i].get("guardrail_tripped", False)
    data[i]["guardrail_pass"] = (guard == TEST_CASES[i]["expectations"]["guardrail_trip"])

    # Check expected tools 
    data[i]["tool_pass"]= False
    if TEST_CASES[i]["expectations"]["expected_tools"]:
        for entry in TEST_CASES[i]["expectations"]["expected_tools"]:
            if entry in data[i]["tools_called"]:
                data[i]["tool_pass"]=True
    else:
        data[i]["tool_pass"]=(TEST_CASES[i]["expectations"]["expected_tools"]==data[i]["tools_called"])
 
    
    # Check sources if relevant

    if TEST_CASES[i]["expectations"]["notice_ids"]:
        #Default is fail
        data[i]["correct_notices"] = False
        for entry in data[i]["sources"]:
            if entry.get("notice_id") in TEST_CASES[i]["expectations"]["notice_ids"]:
                data[i]["correct_notices"] = True
    else:
        data[i]["correct_notices"] = "N/A"
        

df = pd.DataFrame(data)   

average_latency = df["latency_ms"].mean()
print(f"Average Latency: {average_latency} ms")
    

Average Latency: 48479.780952380956 ms


In [7]:
df

,question,tool_calls,tools_called,sources,latency_ms,answer,category,expected_response_notes,guardrail_pass,tool_pass,correct_notices,guardrail_tripped,guardrail_reason
0,Tell me about the public meeting on beavers,"[{'name': 'web_search', 'input': {'input': 'pu...",[web_search],"[{'is_web_source': True, 'url': 'https://www.b...",26226.4,Could you please clarify if you are interested...,Non-existant meeting,Should not find any matching public notices,True,True,N/A,NaN,NaN
1,Tell me about the city meeting on unicorns sig...,[],[],[],1632.8,"Sorry, that doesn't seem to be related to the ...",Non-existant meeting,Unicorns trip the guardrail,True,True,N/A,True,"The query refers to a city meeting, but the su..."
2,Can I testify at the August 6th Tree Removal H...,"[{'name': 'web_search', 'input': {'input': 'Bo...",[web_search],"[{'is_web_source': True, 'url': 'https://conte...",32664.6,"Yes, you can testify at the Tree Removal Heari...",Public testimony,Valid meeting but no public testimony at this ...,True,False,False,NaN,NaN
3,Can I testify at the August 11th Zoning Board ...,"[{'name': 'web_search', 'input': {'input': 'Bo...",[web_search],"[{'is_web_source': True, 'url': 'https://www.b...",52034.1,"Yes, you can testify at the August 11th Zoning...",Public testimony,Valid meeting and public testimony allowed at ...,True,False,False,NaN,NaN
4,Is the Boston Landmarks Commission meeting hap...,"[{'name': 'web_search', 'input': {'input': 'Bo...","[web_search, web_search]","[{'is_web_source': True, 'url': 'https://www.b...",75044.7,"No, the Boston Landmarks Commission meeting is...",Cancelled meeting,Valid meeting but was cancelled,True,False,False,NaN,NaN
5,Is the August 19th St Botolph area meeting hap...,"[{'name': 'web_search', 'input': {'input': 'Au...",[web_search],"[{'is_web_source': True, 'url': 'https://conte...",22245.4,"Yes, the St. Botolph Area Architectural Conser...",Cancelled meeting,Valid meeting but was cancelled,True,False,False,NaN,NaN
6,What is the august 11th city council committee...,"[{'name': 'web_search', 'input': {'input': 'Au...","[web_search, web_search, web_search, web_search]","[{'is_web_source': True, 'url': 'https://www.b...",161587.0,I couldn't access specific information about t...,Public Notice PDF details,Docket PDF affiliated with NoticeID 16603691,True,False,False,NaN,NaN
7,What is the august 13th city council committee...,"[{'name': 'web_search', 'input': {'input': 'Au...","[web_search, web_search, web_search]","[{'is_web_source': True, 'url': 'https://www.b...",93804.0,There doesn't appear to be a public notice or ...,Public Notice PDF details,Docket PDF affiliated with NoticeID 16600696,True,False,False,NaN,NaN
8,Explain crypto wallets,[],[],[],1231.2,"Sorry, that doesn't seem to be related to the ...",Irrelevant Query,Not relevant to city government,True,True,N/A,True,"The query is about crypto wallets, which is a ..."
9,Write a fun limeric,[],[],[],2576.1,"Sorry, that doesn't seem to be related to the ...",Irrelevant Query,Not relevant to city government,True,True,N/A,True,"The query requests a limerick, which falls und..."


In [8]:
# Visual inspection of the answers
for i in range(len(data)):
    print(f"Answer for Query {i}: {data[i]["answer"]}")



Answer for Query 0: Could you please clarify if you are interested in public meetings regarding wildlife management related to beavers, or if you are looking for information about meetings associated with specific places named "Beaver," such as Beaver Place or Beaver Street? This will help in providing the most accurate details.
Answer for Query 1: Sorry, that doesn't seem to be related to the government of the City of Boston.
Answer for Query 2: Yes, you can testify at the Tree Removal Hearing scheduled for August 6th. You have two options for participation: you can provide live or virtual testimony during the hearing, which starts at 10:00 a.m., by raising your hand or using the chat function. Alternatively, if you prefer, you can submit written testimony in advance via email to trees@boston.gov (with "Tree hearing" in the subject line) or by mail to the Tree Warden at the Boston Parks & Recreation Department.

When submitting testimony, be sure to specify the date of the hearing, st

In [13]:
# Investigating differences in latency

web_only = df[df["tools_called"].apply(lambda x: set(x) =={"web_search"})]["latency_ms"]
no_tools = df[df["tools_called"].apply(lambda x: x ==[])]["latency_ms"]

#print(f"Latency when only RAG: {only_rag.mean()} ms")
print(f"Latency when only Web Search: {web_only.mean()} ms")
print(f"Latency when no tools called (guardrails): {no_tools.mean()} ms")

Latency when only Web Search: 63162.60625 ms
Latency when no tools called (guardrails): 1494.7400000000002 ms


In [16]:
print(f"Sources for {2}  {data[2]["sources"]}")

Sources for 2  [{'is_web_source': True, 'url': 'https://content.boston.gov/departments/parks-and-recreation/how-request-tree-removal-hearing'}, {'is_web_source': True, 'url': 'https://www.boston.gov/public-notices/14831'}, {'is_web_source': True, 'url': 'https://www.boston.gov/departments/landmarks-commission/landmarks-and-historic-districts-2026-public-hearing-dates'}, {'is_web_source': True, 'url': 'https://www.boston.gov/departments/parks-and-recreation/participate-public-tree-ordinance-process'}, {'is_web_source': True, 'url': 'https://content.boston.gov/departments/landmarks-commission/landmarks-and-historic-districts-2025-and-2026-public-hearing'}, {'is_web_source': True, 'url': 'https://www.boston.gov/public-notices/16407486'}, {'is_web_source': True, 'url': 'https://www.boston.gov/public-notices?field_contact_target_id=846&title='}, {'is_web_source': True, 'url': 'https://www.boston.gov/sites/default/files/file/2021/07/Parksandrec07212021162808.pdf'}, {'is_web_source': True, 'u